<a href="https://githubtocolab.com/gee-community/geemap/blob/master/docs/notebooks/12_zonal_statistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

Uncomment the following line to install [geemap](https://geemap.org) if needed.

In [1]:
#last edited 7/24, Subi Nair

# organize for list of huc8s across california, work on aggregation for each precip data
!pip install geemap -q
!pip install cartopy scipy -q
!pip install pycrs -q
!pip install geopandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 57.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import geemap
import ee
import pycrs
import io
import requests
import time
import geopandas as gpd
import cartopy
import glob
import pandas as pd
import shutil
import getpass
import zipfile
from google.colab import drive, files

In [5]:
proj_id = getpass.getpass()
ee.Authenticate()
#This way you can copy/paste your project ID
ee.Initialize(project=str(proj_id))

#print(geemap.__version__)
drive.mount('/content/drive')

··········
Mounted at /content/drive


In [9]:
!git clone https://github.com/watrs-csumb/huc8basin-water-balance.git

Cloning into 'huc8basin-water-balance'...
remote: Enumerating objects: 1211, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 1211 (delta 14), reused 15 (delta 9), pack-reused 1185 (from 2)
Receiving objects: 100% (1211/1211), 116.22 MiB | 25.46 MiB/s, done.
Resolving deltas: 100% (159/159), done.
Updating files: 100% (974/974), done.


Please take a moment to verify the units of each data product.



*   gridmet: mm
*   chirps: mm
*   prism: mm
*   nclimgrid: mm
*   nldas-2: mm
*   AgEra-5: kg/m^2
*   rdpa: mm


In [7]:
## dictionary of variables FORMAT: [feature collection, precip band, pixels, units]

## edit dictionary
variable_dict  = {
    'gridmet':['IDAHO_EPSCOR/GRIDMET','pr',4638.3, 'mm'],
    'chirps':['UCSB-CHG/CHIRPS/DAILY','precipitation',5566, 'mm'],
    'prism':['projects/sat-io/open-datasets/OREGONSTATE/PRISM_800_DAILY', 'ppt',800, 'mm'],
    'nclimgrid':['projects/climate-engine-pro/assets/noaa-ncei-nclimgrid/daily', 'precip',4630, 'mm'],
    'nldas-2':['NASA/NLDAS/FORA0125_H002','total_precipitation',13915, 'mm'],
    'AgEra-5':['projects/climate-engine-pro/assets/ce-ag-era5-v2/daily','Precipitation_Flux',9600, 'kg/m^2'],
    'rdpa':['projects/climate-engine-pro/assets/ce-rdpa-daily','precip',10000, 'mm'], #starts 2003
    #'acis':['projects/climate-engine-pro/assets/noaa-nrcc-acis-nn/daily', 'precip', 5000, 'in']
}


## Function to select precipitation data source for date-range.

def select_precip_data(dataset_name, start_date, end_date):

  image_col_i = variable_dict[dataset_name][0]
  band_i = variable_dict[dataset_name][1]
  scale_i = variable_dict[dataset_name][2]
  unit_i = variable_dict[dataset_name][3]

  # assign the selected image collection to a variable
  selected_imgcol = ee.ImageCollection(image_col_i).filter(
      ee.Filter.date(start_date, end_date)
  )

  precip_collection = selected_imgcol.select(band_i)

  # convert inches, for acis dataset
  # rename(band_i) because the multiplication will cause earth engine to drop the property metadata
  # creates an entirely new object instead of modifying in place so you need to tell it to keep the name

  if unit_i == 'in':
      # We chain .multiply() AND .rename() inside the same lambda expression
      precip_collection = precip_collection.map(
          lambda image: image.multiply(25.4).rename(band_i)
      )

  return precip_collection, scale_i, band_i


# This is the function to reduce the image collection
# and return a single image that can be clipped
# squashes the dataset through time

def reduce_precip_image(image_collection):
  # Reduce the ImageCollection to a single Image (e.g., sum of precipitation)
  precip_image = image_collection.sum()

  #return the reduced image so it can be clipped later for the map
  return precip_image

In [8]:
# CSV IMPORT FOR HUC CODES
# use pandas to import the csv

# format should be huc column name and 1 huc per row, single column at the moment
# use 'isinstance' to check for list or csv

def input_huc_codes(input_codes) :
    if isinstance(input_codes, list):
        print("Detected input type: LIST")
        return pd.DataFrame(input_codes)

    # Check if the input is a string // csv file path
    elif isinstance(input_codes, str):
        print("Detected input type: CSV FILE")
        return pd.read_csv(input_codes)

    else:
        raise TypeError("Unsupported input type. Please provide a list or a CSV path string.")

In [11]:
## Huc testing options below, for small lists to full files
#target_hucs = ["18040003"] # Define multiple target HUCs
#target_hucs_in_CA = ["18040003","18090202","18020109"]
#huc_list = ['18040014', '18020126', '17110005', '17100302']

#huc list from github
huc_list_csv = '/content/huc8basin-water-balance/data/metadata/Selected_basins.csv'

def get_multiple_hucs(huc_input):
    # Detect input type and load into a Pandas DataFrame
    if isinstance(huc_input, list):
        df = pd.DataFrame(huc_input)
    elif isinstance(huc_input, str):
        df = pd.read_csv(huc_input)
    else:
        raise TypeError("Please provide a list or a CSV path string.")

    # Extract the HUC IDs into a flat Python list for Earth Engine
    # This assumes your list contains strings, or your CSV has a column with HUC codes
    # .iloc[:, 0] grabs the very first column automatically
    huc_list = df.iloc[:, 0].astype(str).tolist()

    # Load the USGS HUC08 dataset
    huc_collection = ee.FeatureCollection("USGS/WBD/2017/HUC08")

    # Filter for all HUCs present in your input list
    filtered_hucs = huc_collection.filter(ee.Filter.inList("huc8", huc_list))

    #print(filtered_hucs)

    return filtered_hucs #a feature collection

filtered_collection = get_multiple_hucs(huc_list_csv) # Use the function to get a FeatureCollection of multiple HUCs




In [12]:
#This loop cycles through dataset names and their corresponding properties (image collection and unit).
for dataset_i, (image_col_i, band_i, scale_i, unit_i) in variable_dict.items():
    print(f"Dataset: {dataset_i}, Image Collection: {image_col_i}, Unit: {unit_i}")

#unpack the return values into three variables
#then run reduce_precip_image
# THIS IS THE IMPORTANT BIT FOR THE LATER PIECES WITH MONHTLY AND EXPORTING

pr_collection, scale, band = select_precip_data(dataset_i, '2000-01-01', '2025-12-31')

#This is for mapping
precip_image = reduce_precip_image(pr_collection)

# The following code is mostly useful for visualizations at this stage

def clip_to_huc(feature):
    # to retain the huc ID
    huc_id = feature.get('huc8')

    # Clip the main precipitation image to this specific HUC feature
    clipped = precip_image.clip(feature.geometry())

    # Optional: Copy the HUC ID property onto the clipped image so you know which is which
    return clipped.set('huc8', huc_id)

#map the clipping function over the filtered collection
# Clipping here primarily serves purpose for displaying on map later, the daily stats and monthly stats functions -
# - both use reduce regions to do this instead.

clipped_precip_collection = filtered_collection.map(clip_to_huc)

# Print out to make sure that it worked using get info instead of the standard print which will not give you info
# Additionally the clipping setup here can help us grab each huc as it is processed so we can check if something goes wrong with a huc
# or if a huc is missing

collection_info = clipped_precip_collection.getInfo()

for item in collection_info['features']:
    # Grab the HUC ID we saved in the metadata
    saved_huc_id = item['properties']['huc8']
    print(f"✅ Success! Found processed data for HUC ID: {saved_huc_id}")

Dataset: gridmet, Image Collection: IDAHO_EPSCOR/GRIDMET, Unit: mm
Dataset: chirps, Image Collection: UCSB-CHG/CHIRPS/DAILY, Unit: mm
Dataset: prism, Image Collection: projects/sat-io/open-datasets/OREGONSTATE/PRISM_800_DAILY, Unit: mm
Dataset: nclimgrid, Image Collection: projects/climate-engine-pro/assets/noaa-ncei-nclimgrid/daily, Unit: mm
Dataset: nldas-2, Image Collection: NASA/NLDAS/FORA0125_H002, Unit: mm
Dataset: AgEra-5, Image Collection: projects/climate-engine-pro/assets/ce-ag-era5-v2/daily, Unit: kg/m^2
Dataset: rdpa, Image Collection: projects/climate-engine-pro/assets/ce-rdpa-daily, Unit: mm
✅ Success! Found processed data for HUC ID: 17040101
✅ Success! Found processed data for HUC ID: 17110006
✅ Success! Found processed data for HUC ID: 17110016
✅ Success! Found processed data for HUC ID: 17100103
✅ Success! Found processed data for HUC ID: 17010302
✅ Success! Found processed data for HUC ID: 17010301
✅ Success! Found processed data for HUC ID: 17040205
✅ Success! Found

In [20]:
# Calculate daily stats
# Function to calculate the daily average over the basin area
# This will time out if you use the API for more than a few
# squash the data across space

def calculate_daily_stats(img_col, img_scale, img_band):

  def process_single_image(img):

      # Ask the server for the actual bands present in THIS specific daily image file
      band_names = img.bandNames()
      has_band = band_names.contains(img_band)

      # Grab the date tracking string
      date_string = img.get('system:time_start')

      # If the image is empty or has 0 bands, we return a blank placeholder
      # instead of allowing reduceRegions to crash the Drive task.
      # this is an issue for datasets like RDPA
      # which may only have images for some of the years in the total default range

      protected_features = ee.Algorithms.If(
          has_band,
          # IF TRUE: Execute reduction as normal
          img.reduceRegions(
              reducer=ee.Reducer.mean(),
              collection=filtered_collection,
              scale=img_scale,
              maxPixelsPerRegion=1e9
          ).map(lambda huc_feature: ee.Feature(None, {
              'date': date_string,
              'huc8': huc_feature.get('huc8'),
              'precip_val': huc_feature.get('mean')
          })),
          # IF FALSE: Return a completely empty feature collection shell
          ee.FeatureCollection([])
      )

      return ee.FeatureCollection(protected_features)

  # Map the calculation over every daily node and flatten the result
  return ee.FeatureCollection(img_col.map(process_single_image)).flatten()

In [17]:
## Calculating Monthly Stats
## squash the data across space

def calculate_monthly_stats(img_col, img_scale, img_band, start_date, end_date):
    """
    Groups daily precipitation data into monthly sums, then calculates
    the spatial mean for each HUC feature across the entire time range.
    """
    # 1. Parse dates into Earth Engine date objects
    ee_start = ee.Date(start_date)
    ee_end = ee.Date(end_date)

    # Calculate total number of months between start and end dates
    n_months = ee_end.difference(ee_start, 'month').round()
    month_indices = ee.List.sequence(0, n_months.subtract(1))

    # 2. Inner helper function to safely create monthly sum images
    def make_monthly_sum(index):
        current_month_start = ee_start.advance(ee.Number(index), 'month')

        # Filter daily collection down to just this single month block
        monthly_ic = img_col.filterDate(
            current_month_start,
            current_month_start.advance(1, 'month')
        )

        # CRITICAL FIX: Verify the daily collection has images BEFORE summing
        collection_size = monthly_ic.size()

        def true_case():
            # If images exist, isolate the target band and calculate the sum
            sum_img = monthly_ic.select(img_band).sum()
            return sum_img.set({
                'is_valid': 1,
                'system:time_start': current_month_start.millis()
            })

        def false_case():
            # Return a blank dummy image with a flag if no daily data exists
            return ee.Image.constant(0).rename(img_band).set({
                'is_valid': 0,
                'system:time_start': current_month_start.millis()
            })

        # Evaluate the server-side condition safely
        final_image = ee.Algorithms.If(collection_size.gt(0), true_case(), false_case())
        return ee.Image(final_image)

    # Generate collection and strip out the flagged empty months
    monthly_collection = ee.ImageCollection(month_indices.map(make_monthly_sum))
    valid_monthly_collection = monthly_collection.filter(ee.Filter.eq('is_valid', 1))

    # 3. Inner helper function to execute spatial reductions
    def process_single_month(month_img):
        stats = month_img.reduceRegions(
            reducer=ee.Reducer.mean(),
            collection=filtered_collection, # Global HUC collection variable
            scale=img_scale,
            maxPixelsPerRegion=1e9
        )

        month_string = month_img.get('system:time_start')

        def attach_month(huc_feature):
            return ee.Feature(None, {
                'date': month_string,
                'huc8': huc_feature.get('huc8'),
                'precip_val': huc_feature.get('mean')
            })

        return stats.map(attach_month)

    # Map across valid months and flatten features
    raw_features = valid_monthly_collection.map(process_single_month).flatten()

    # 4. Strict Schema Enforcement for clean exports
    def enforce_headers(feature):
        return ee.Feature(None, {
            'date': ee.Number(feature.get('date')),
            'huc8': ee.String(feature.get('huc8')),
            'precip_val': ee.Number(feature.get('precip_val'))
        })

    return raw_features.map(enforce_headers)


In [18]:
## EXPORTER FUNCTION SECTION
## WILL TRY A DIRECT DOWNLOAD FIRST
# can be condensed later, some repeated code for daily and monthly

def export_huc_precipitation_data(pr_collection, scale, band, dataset_id, start_date='2000-01-01',
                                  end_date='2025-12-31', timeframe_choice='both'):
    """
    Wrapped exporter function that calculates zonal statistics for HUC features,
    cleans the data tables, pivots formats, and packages them into localized files.

    Parameters:
      timeframe_choice: 'both' (default), 'daily', or 'monthly'
    """

    # Map the passed input variables into your existing script variables
    start_d = start_date
    end_d = end_date
    dataset_name = dataset_id # Label used for file naming
    local_zip_filename = f'{dataset_name}_{timeframe_choice}_precipitation_package.zip'

    # We will collect dataframes here to process them together in the ZIP loop
    # timeframes refer to daily or monthly
    timeframes_to_process = {}

    # Evaluate conditional choices based on parameter input
    run_daily = timeframe_choice.lower() in ['daily', 'both']
    run_monthly = timeframe_choice.lower() in ['monthly', 'both']


    ############# RUN CALCULATIONS & FETCH DATA #################

    # PART 1: RUN DAILY STATS
    ###########################
    if run_daily:
        print("\n--- Processing DAILY Statistics ---")
        daily_features = calculate_daily_stats(pr_collection, scale, band)
        downloadable_daily = daily_features.select(
            propertySelectors=['date', 'huc8', 'precip_val'],
            retainGeometry=False
        )

        # try through the API first
        df_daily_long = pd.DataFrame()
        print("Attempting to fetch daily data directly from Earth Engine servers...")
        try:
            download_url_daily = downloadable_daily.getDownloadURL(
                filetype="CSV",
                selectors=['date', 'huc8', 'precip_val'],
                filename="daily_stats"
            )
            response = requests.get(download_url_daily)
            if response.status_code == 200 and len(response.text.strip()) > 0:
                df_daily_long = pd.read_csv(io.StringIO(response.text))
                print("Direct download successful for daily data")
        except Exception as e:
            print(f"Direct daily download failed or timed out: {e}")

        #switch to a server task if that doesn't work

        if df_daily_long.empty:
            print("Daily dataset is too large. Switching to Google Drive batch export for daily...")
            target_drive_folder = 'exported_precip'
            file_prefix_daily = f'HUC_Daily_{dataset_name}_Batch'

            task_daily = ee.batch.Export.table.toDrive(
                collection=downloadable_daily,
                description=file_prefix_daily,
                fileFormat='CSV',
                folder=target_drive_folder,
                fileNamePrefix=file_prefix_daily,
                selectors=['date', 'huc8', 'precip_val'],
            )
            task_daily.start()

            ## check status every 30 minutes and report back
            print("Cloud task started. Monitoring daily export status...")
            while task_daily.active():
                time.sleep(30)
                print(f"Daily task status: {task_daily.status()['state']}...")

            if task_daily.status()['state'] == 'COMPLETED':
                print("Task complete! Loading daily file from Google Drive...")
                drive_csv_path = f'/content/drive/MyDrive/{target_drive_folder}/{file_prefix_daily}.csv'
                df_daily_long = pd.read_csv(drive_csv_path)
            else:
                print(f"Earth Engine daily task failed with status: {task_daily.status()['state']}")

        if not df_daily_long.empty:
            timeframes_to_process['daily'] = df_daily_long

    # PART 2: RUN MONTHLY STATS
    #############################

    if run_monthly:
        print("\n--- Processing MONTHLY Statistics ---")
        monthly_features = calculate_monthly_stats(pr_collection, scale, band, start_d, end_d)
        downloadable_monthly = monthly_features.select(
            propertySelectors=['date', 'huc8', 'precip_val'],
            retainGeometry=False
        )

        df_monthly_long = pd.DataFrame()

        # API attempt

        print("Attempting to fetch monthly data directly from Earth Engine servers...")
        try:
            download_url_monthly = downloadable_monthly.getDownloadURL(
                filetype="CSV",
                selectors=['date', 'huc8', 'precip_val'],
                filename="monthly_stats"
            )
            response = requests.get(download_url_monthly)
            if response.status_code == 200 and len(response.text.strip()) > 0:
                df_monthly_long = pd.read_csv(io.StringIO(response.text))
                print("Direct download successful for monthly data")
        except Exception as e:
            print(f"Direct monthly download failed or timed out: {e}")

        # switching to server task if that doesn't work

        if df_monthly_long.empty:
            print("Monthly dataset is too large. Switching to Google Drive batch export for monthly...")
            target_drive_folder = 'exported_precip'
            file_prefix_monthly = f'HUC_Monthly_{dataset_name}_Batch'

            task_monthly = ee.batch.Export.table.toDrive(
                collection=downloadable_monthly,
                description=file_prefix_monthly,
                fileFormat='CSV',
                folder=target_drive_folder,
                fileNamePrefix=file_prefix_monthly,
                selectors=['date', 'huc8', 'precip_val'],
            )
            task_monthly.start()

            print("Cloud task started. Monitoring monthly export status...")
            while task_monthly.active():
                time.sleep(30)
                print(f"Monthly task status: {task_monthly.status()['state']}...")

            if task_monthly.status()['state'] == 'COMPLETED':
                print("Task complete! Loading monthly file from Google Drive...")
                drive_csv_path = f'/content/drive/MyDrive/{target_drive_folder}/{file_prefix_monthly}.csv'
                df_monthly_long = pd.read_csv(drive_csv_path)
            else:
                print(f"Earth Engine monthly task failed with status: {task_monthly.status()['state']}")

        if not df_monthly_long.empty:
            timeframes_to_process['monthly'] = df_monthly_long

    # CLEAN, PIVOT, AND ZIP PACKAGING
    if timeframes_to_process:
        print(f"\nBuilding your ZIP archive: '{local_zip_filename}'...")

        # to determine if subfolders are needed
        use_subfolders = len(timeframes_to_process) > 1

        with zipfile.ZipFile(local_zip_filename, 'w', zipfile.ZIP_DEFLATED) as zip_file:
            for timeframe, df_long in timeframes_to_process.items():
                print(f"\nProcessing raw {timeframe} data into individual HUC files...")

                # --- THE PANDAS FIXES ---
                # Start with long data and then pivot to wide columns
                # Drop completely empty rows to protect the index
                # make a copy to avoid SettingWithCopyWarning
                # since it would point to the original df_long
                df_long = df_long.dropna(subset=['date', 'precip_val'])
                df_long = df_long.copy()

                # Enforce clean Pandas datetime format using the raw millisecond numbers
                df_long['date'] = pd.to_datetime(df_long['date'], unit='ms').dt.strftime('%Y-%m-%d')

                # Clean out overlapping polygon duplicates before reshaping
                # This keeps the first of the duplicates. At this stage we only need unique huc values.
                df_long = df_long.drop_duplicates(subset=['date', 'huc8'], keep='first')

                # Pivot the table from "long" format to "wide" format
                df_wide = df_long.pivot(index='date', columns='huc8', values='precip_val')
                # When Pandas pivots a table, it turns your date column into the "index"
                # This line pushes date back out into a regular data column
                df_wide = df_wide.reset_index()
                df_wide = df_wide.sort_values('date').reset_index(drop=True)

                huc_columns = [col for col in df_wide.columns if col != 'date']
                print(f"Packing {len(huc_columns)} {timeframe} HUC files into the zip...")

                # Write individual HUC files into the shared zip
                for huc_code in huc_columns:
                    df_single_huc = df_wide[['date', huc_code]].copy()
                    df_single_huc.columns = ['date', 'precipitation_mm']
                    df_single_huc = df_single_huc.dropna().reset_index(drop=True)

                    if not df_single_huc.empty:
                        csv_text = df_single_huc.to_csv(index=False)

                    # Unique name paths inside the ZIP: e.g., "daily/huc_17110005_gridmet_daily.csv"
                    # Choose file path based on run choice
                    if use_subfolders:
                        file_name = f'{timeframe}/huc_{huc_code}_{dataset_name}_{timeframe}_precipitation.csv'
                    else:
                        file_name = f'huc_{huc_code}_{dataset_name}_{timeframe}_precipitation.csv'

                    zip_file.writestr(file_name, csv_text)

        print(f"\nStreaming {local_zip_filename} straight to your machine's Downloads folder...")
        files.download(local_zip_filename)
    else:
        print("\nExport sequence failed because no data could be retrieved for either timeframe.")


In [19]:
#Both
#export_huc_precipitation_data(pr_collection, scale, band, dataset_i)

#Daily
#export_huc_precipitation_data(pr_collection, scale, band, dataset_i, timeframe_choice='daily')

#Monthly
export_huc_precipitation_data(pr_collection, scale, band, dataset_i, timeframe_choice='monthly',
                              start_date='2000-01-01', end_date='2025-12-31')




--- Processing MONTHLY Statistics ---
Attempting to fetch monthly data directly from Earth Engine servers...
Direct download successful for monthly data

Building your ZIP archive: 'gridmet_monthly_precipitation_package.zip'...

Processing raw monthly data into individual HUC files...
Packing 81 monthly HUC files into the zip...

Streaming gridmet_monthly_precipitation_package.zip straight to your machine's Downloads folder...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
## OPTIONAL: Map all of the clipped images as a mosaic
# can be helpful to verify the hucs

# Set visualization parameters.
HUC_vis = {
    'color': '0000FFFF',
    'width': 1,
    'lineType': 'solid',
    'fillColor': '000000'
}

precip_vis = {
    'min': 0,
    'max': 100,
    'palette': ['#F0F8FF', '#ADD8E6', '#00BFFF', '#1E90FF', '#00008B'],
}

# Add the original sum precipitation image
#Map.add_layer(precip_image, precip_vis, 'Sum Precipitation')

# View selected basin(s)
Map= geemap.Map(center = (39.833,-98.583), zoom = 4)
Map.add_layer(filtered_collection, HUC_vis, "Selected basin")

clipped_mosaic = ee.ImageCollection(clipped_precip_collection).mosaic()

Map.add_layer(clipped_mosaic, precip_vis, 'Clipped Sum Precipitation')
Map

Map(center=[39.833, -98.583], controls=(WidgetControl(options=['position', 'transparent_bg'], position='toprig…